In [ ]:
import sys, os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

import utils as ut
import meshio
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
datadir = "./"
inname = "Nicoya_interface.e"
outname = "test.geo"
geofile = datadir + outname


In [ ]:
mesh = meshio.read(datadir + inname)
points = mesh.points  # shape (n_points, 3)
print(mesh.cells)

In [ ]:
# Create DataFrame
df = pd.DataFrame(points, columns=["x", "y", "z"])

# Print ranges
ranges = df.agg(["min", "max"])
print(ranges)

In [ ]:
df['lon'], df['lat'] = ut.ckm2LLd(df['x'], df['y'], -84, 7, -45)

In [ ]:
# Import GNSS data
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
# obs_disp_name = "Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_disp_name = "Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed
# note that the height is in m, Dt in years, the original velocity data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
data = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                          'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                          'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
data['lon'] = -1*data['lon'] # convert to east positive, as the original data is west positive

data['x'], data['y']  = ut.LL2ckmd(data['lon'], data['lat'], -84, 7, 45)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(df['lon'], df['lat'], s=10, color='blue', marker='o')  # solid dots
plt.scatter(data['lon'], data['lat'], s=10, color='red', marker='^')  # solid dots
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.axis("equal")  # equal scaling for x/y
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df['x'], df['y'], s=10, color='blue', marker='o')  # solid dots
plt.scatter(data['x'], data['y'], s=10, color='red', marker='^')  # solid dots
plt.xlabel("x rot")
plt.ylabel("y rot")
plt.axis("equal")  # equal scaling for x/y
plt.grid(True)
plt.show()

In [ ]:
# ut.exo_fault_to_geo(
#     datadir + inname,
#     geo_file=geofile,
#     lc=2500,          # mesh size
#     fault_id=7        # match your .geo numbering for fault
# )

In [ ]:
# Define sep
sep = "//+"

# Define the cubiod boundary points, in meters
xmin, xmax = -1000e3, 1000e3
ymin, ymax = -1000e3, 1000e3
zmin, zmax = -400e3, 0.0
print(xmin, xmax, ymin, ymax, zmin, zmax)

x0, y0 = 130e3, 350e3  # offset for x and y coordinates

# write to the .geo file 
geo_content = f"""\
{sep}
xmin = {xmin};
xmax = {xmax};
ymin = {ymin};
ymax = {ymax};
zmin = {zmin};
zmax = {zmax}; 

"""

# Write to the file
with open(geofile, "w") as f:
    f.write(geo_content)


# define the anomaly??

# Define the mesh sizes for different regions, in meters
dx = 200e3
dx_fault = 5e3
dx_fault_coarse = 20e3
dx_anomaly = 5e3
dx_anomaly_coarse = 20e3

geo_content = f"""\
{sep}
dx = {dx};
dx_fault = {dx_fault};
dx_fault_coarse = {dx_fault_coarse};
dx_anomaly = {dx_anomaly};
dx_anomaly_coarse = {dx_anomaly_coarse};

"""

# append to the file
with open(geofile, "a") as f:
    f.write(geo_content)



In [ ]:
# ut.exo_fault_to_points(datadir + inname, geo_file=geofile, dx_str="dx_fault_coarse")

# def exo_fault_to_points_with_dx(infile, break1_list, break2_list,
#                                 geo_file=None, coarse_str="dx_fault_coarse", fine_str="dx_fault"):
#     mesh = meshio.read(infile)
#     points = mesh.points

#     if geo_file is None:
#         geo_file = str(Path(infile).with_suffix(".geo"))

#     # Build a set of all point IDs that are in segment 2 ranges
#     seg2_points = set()
#     for b1, b2 in zip(break1_list, break2_list):
#         seg2_points.update(range(b1, b2 + 1))

#     with open(geo_file, "a") as f:
#         for i, (x, y, z) in enumerate(points, start=1):
#             dx_str = fine_str if i in seg2_points else coarse_str
#             f.write(f"Point({i}) = {{{x}, {y}, {z}, {dx_str}}};\n")

#     print(f"Wrote {geo_file} with {len(points)} points "
#           f"({len(seg2_points)} points got '{fine_str}')")

In [ ]:
# line_id = 1
# expr = "1, 2, 5:2:287"
# ut.append_spline_to_geo(geofile, line_id, expr)

# line_id = 1
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "1, 2, 5, 7, 9")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "9:2:177")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "177:2:287")

# line_id = 2
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "4, 3, 6, 8, 10")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "10:2:178")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "178:2:288")

# line_id = 3
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "290, 289, 291:293")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "293:377")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "377:432")

# line_id = 4
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "434, 433, 435:437")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "437:521")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "521:576")

# line_id = 5
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "578, 577, 579:581")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "581:665")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "665:720")

# line_id = 6
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "722, 721, 723:725")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "725:809")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "809:864")

# line_id = 7
# ut.append_spline_to_geo(geofile, (line_id-1)*3+1, "866, 865, 867:869")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+2, "869:953")
# ut.append_spline_to_geo(geofile, (line_id-1)*3+3, "953:1008")

# Your lists
# break points between segments 1 and 2
break1_list = [
    9, 10, 293, 437, 581, 725, 869,
    1013, 1157, 1301, 1445, 1589, 1733, 1877, 2021, 2165, 2309, 2453, 2597, 2741, 2885, 3029, 3173,
    3317, 3461, 3605, 3749, 3893, 4037, 4181, 4325, 4469, 4613, 4757, 4901, 5045, 5189, 5333, 5477,
    5621, 5765, 5909, 6053, 6197, 6341, 6485, 6629, 6773, 6917, 7061, 7205, 7349, 7493, 7637, 7781,
    7925, 8069, 8213, 8357, 8501, 8645, 8789, 8933, 9077, 9221, 9365, 9509
]
# break points between segments 2 and 3
break2_list = [
    177, 178, 377, 521, 665, 809, 953,
    1097, 1241, 1385, 1529, 1673, 1817, 1961, 2105, 2249, 2393, 2537, 2681, 2825, 2969, 3113, 3257,
    3401, 3545, 3689, 3833, 3977, 4121, 4265, 4409, 4553, 4697, 4841, 4985, 5129, 5273, 5417, 5561,
    5705, 5849, 5993, 6137, 6281, 6425, 6569, 6713, 6857, 7001, 7145, 7289, 7433, 7577, 7721, 7865,
    8009, 8153, 8297, 8441, 8585, 8729, 8873, 9017, 9161, 9305, 9449, 9593
]
# ending points of segment 3
end3_list = [
    287, 288, 432, 576, 720, 864, 1008,
    1152, 1296, 1440, 1584, 1728, 1872, 2016, 2160, 2304, 2448, 2592, 2736, 2880, 3024, 3168, 3312,
    3456, 3600, 3744, 3888, 4032, 4176, 4320, 4464, 4608, 4752, 4896, 5040, 5184, 5328, 5472, 5616,
    5760, 5904, 6048, 6192, 6336, 6480, 6624, 6768, 6912, 7056, 7200, 7344, 7488, 7632, 7776, 7920,
    8064, 8208, 8352, 8496, 8640, 8784, 8928, 9072, 9216, 9360, 9504, 9648
]
# start points of segment 1
start1_list = [1, 4] + [x + 2 for x in end3_list[1:-1]]

print(len(break1_list), len(break2_list), len(end3_list), len(start1_list))
print(start1_list[-5:])

In [ ]:
def exo_fault_to_points_with_dx(infile, break1_list, break2_list,
                                geo_file=None,
                                coarse_str="dx_fault_coarse",
                                fine_str="dx_fault"):
    mesh = meshio.read(infile)
    points = mesh.points

    if geo_file is None:
        geo_file = str(Path(infile).with_suffix(".geo"))

    seg2_points = set()
    for idx, (b1, b2) in enumerate(zip(break1_list, break2_list), start=1):
        if idx in (1, 2):
            seg2_points.update(range(b1, b2 + 1, 2))  # step=2 for lines 1 & 2
        else:
            seg2_points.update(range(b1, b2 + 1))      # normal contiguous

    with open(geo_file, "a") as f:
        for i, (x, y, z) in enumerate(points, start=1):
            dx_str = fine_str if i in seg2_points else coarse_str
            f.write(f"Point({i}) = {{{x-x0}, {y-y0}, {z}, {dx_str}}};\n")

    print(f"Wrote {geo_file} with {len(points)} points "
          f"({len(seg2_points)} points got '{fine_str}')")
    
    return len(points)

In [ ]:
npts = exo_fault_to_points_with_dx(
    infile=datadir + inname,
    break1_list=break1_list,
    break2_list=break2_list,
    geo_file=geofile,
    coarse_str="dx_fault_coarse",
    fine_str="dx_fault"
)

In [ ]:
# Start line_id
start_line_id = 1

# Loop through all entries
for i, (b1, b2, e3) in enumerate(zip(break1_list, break2_list, end3_list)):
    line_id = start_line_id + i
    base_id = (line_id - 1) * 3 + 1

    if i == 0:
        # Segment 1 expression
        seg1_expr = f"1, 2, 5:2:9"
        # Segment 2 expression
        seg2_expr = f"9:2:177"
        # Segment 3 expression
        seg3_expr = f"177:2:287"

    elif i == 1:    
        # Segment 1 expression
        seg1_expr = f"4, 3, 6:2:10"
        # Segment 2 expression
        seg2_expr = f"10:2:178"
        # Segment 3 expression
        seg3_expr = f"178:2:288"

    else:    
        # Segment 1 expression
        seg1_expr = f"{b1-3}, {b1-4}, {b1-2}:{b1}"
        # Segment 2 expression
        seg2_expr = f"{b1}:{b2}"
        # Segment 3 expression
        seg3_expr = f"{b2}:{e3}"

    ut.append_spline_to_geo(geofile, base_id, seg1_expr)
    ut.append_spline_to_geo(geofile, base_id + 1, seg2_expr)
    ut.append_spline_to_geo(geofile, base_id + 2, seg3_expr)

In [ ]:
# update point and line count
start_point_id = npts + 1
start_line_id += base_id+1
print(start_line_id)

In [ ]:
# === Step 1: Create vertical lines ===
vertical_line_start_ids = []
vertical_line_break1_ids = []
vertical_line_break2_ids = []
vertical_line_end3_ids = []

with open(geofile, "a") as f:
    # start1_list
    for i in range(len(start1_list) - 1):
        start_line_id += 1
        vertical_line_start_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{start1_list[i]}, {start1_list[i+1]}}};\n")

    # break1_list
    for i in range(len(break1_list) - 1):
        start_line_id += 1
        vertical_line_break1_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break1_list[i]}, {break1_list[i+1]}}};\n")

    # break2_list
    for i in range(len(break2_list) - 1):
        start_line_id += 1
        vertical_line_break2_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{break2_list[i]}, {break2_list[i+1]}}};\n")

    # end3_list
    for i in range(len(end3_list) - 1):
        start_line_id += 1
        vertical_line_end3_ids.append(start_line_id)
        f.write(f"Line({start_line_id}) = {{{end3_list[i]}, {end3_list[i+1]}}};\n")

# print(vertical_line_start_ids[:3])
# print(vertical_line_break1_ids[:3])
# print(vertical_line_break2_ids[:3])
# print(vertical_line_end3_ids[:3])

# === Step 2: Build curve loops and surfaces (now includes last set) ===
surface_id = 1
curve_loop_id = 1

with open(geofile, "a") as f:
    for i in range(len(break1_list)-1):  # loop until second-to-last
        base_id = i * 3 + 1
        next_base_id = (i + 1) * 3 + 1

        # --- Segment 1 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id}, {vertical_line_break1_ids[i]}, -{next_base_id}, -{vertical_line_start_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # --- Segment 2 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id+1}, {vertical_line_break2_ids[i]}, -{next_base_id+1}, -{vertical_line_break1_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

        # --- Segment 3 surface ---
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{base_id+2}, {vertical_line_end3_ids[i]}, -{next_base_id+2}, -{vertical_line_break2_ids[i]}}};\n")
        f.write(f"Surface({surface_id}) = {{{curve_loop_id}}};\n")
        curve_loop_id += 1
        surface_id += 1

In [ ]:
### ADD points, and surfaces connecting the boundary at ymax
# --- Step 1: Create ymax points and track their IDs ---
ymax_point_ids = []  # IDs of points at ymax
vertical_edges_to_ymax = []  # Lines from original point -> ymax point
horizontal_ymax_lines = []  # Lines connecting consecutive ymax points

with open(geofile, "a") as f:
    # Create ymax points
    for i, pid in enumerate(start1_list, start=start_point_id):
        x, _, z = points[pid - 1]  # pid is 1-based, array is 0-based
        f.write(f"Point({i}) = {{{x-x0}, ymax, {z}, dx}};\n")
        ymax_point_ids.append(i)

# --- Step 2: Create vertical lines from ymax to original ---
with open(geofile, "a") as f:
    for orig_pid, ymax_pid in zip(start1_list, ymax_point_ids):
        start_line_id += 1
        f.write(f"Line({start_line_id}) = {{{ymax_pid}, {orig_pid}}};\n")
        vertical_edges_to_ymax.append(start_line_id)

# --- Step 3: Create horizontal lines between consecutive ymax points ---
with open(geofile, "a") as f:
    for p1, p2 in zip(ymax_point_ids[:-1], ymax_point_ids[1:]):
        start_line_id += 1
        f.write(f"Line({start_line_id}) = {{{p1}, {p2}}};\n")
        horizontal_ymax_lines.append(start_line_id)

# update point and line count
start_point_id += len(start1_list)

# --- Step 4: Create curve loops and surfaces using ymax points ---
with open(geofile, "a") as f:
    for i in range(len(start1_list) - 1):
        # lines in CCW order: bottom -> right vertical -> top -> left vertical
        left_vertical = vertical_edges_to_ymax[i]
        bottom_line = vertical_line_start_ids[i]
        right_vertical = -vertical_edges_to_ymax[i+1]  # reverse orientation
        top_line = -horizontal_ymax_lines[i]  # reverse orientation
        
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{left_vertical}, {bottom_line}, {right_vertical}, {top_line}}};\n")
        f.write(f"Plane Surface({surface_id}) = {{{curve_loop_id}}};\n")

        curve_loop_id += 1
        surface_id += 1

In [ ]:
### ADD points, and surfaces connecting the boundary at ymin
# --- Step 1: Create ymin points and track their IDs ---
ymin_point_ids = []  # IDs of points at ymin
vertical_edges_to_ymin= []  # Lines from original point -> ymin point
horizontal_ymin_lines = []  # Lines connecting consecutive ymin points

with open(geofile, "a") as f:
    # Create ymin points
    for i, pid in enumerate(end3_list, start=start_point_id):
        x, _, z = points[pid - 1]  # pid is 1-based, array is 0-based
        f.write(f"Point({i}) = {{{x-x0}, ymin, {z}, dx}};\n")
        ymin_point_ids.append(i)

# --- Step 2: Create vertical lines from original to ymin ---
with open(geofile, "a") as f:
    for orig_pid, ymin_pid in zip(end3_list, ymin_point_ids):
        start_line_id += 1
        f.write(f"Line({start_line_id}) = {{{orig_pid}, {ymin_pid}}};\n")
        vertical_edges_to_ymin.append(start_line_id)

# --- Step 3: Create horizontal lines between consecutive ymin points ---
with open(geofile, "a") as f:
    for p1, p2 in zip(ymin_point_ids[:-1], ymin_point_ids[1:]):
        start_line_id += 1
        f.write(f"Line({start_line_id}) = {{{p1}, {p2}}};\n")
        horizontal_ymin_lines.append(start_line_id)

# update point and line count
start_point_id += len(end3_list)

# --- Step 4: Create curve loops and surfaces using ymax points ---
with open(geofile, "a") as f:
    for i in range(len(end3_list) - 1):
        # lines in CCW order: bottom -> right vertical -> top -> left vertical
        left_vertical = vertical_edges_to_ymin[i]
        bottom_line = horizontal_ymin_lines[i] 
        right_vertical = -vertical_edges_to_ymin[i+1]  # reverse orientation
        top_line = -vertical_line_end3_ids[i] # reverse orientation
        
        f.write(f"Curve Loop({curve_loop_id}) = "
                f"{{{left_vertical}, {bottom_line}, {right_vertical}, {top_line}}};\n")
        f.write(f"Plane Surface({surface_id}) = {{{curve_loop_id}}};\n")

        curve_loop_id += 1
        surface_id += 1

In [ ]:
### ADD points, and surfaces connecting the boundary at zmin
# Last element from break lists
b1 = break1_list[-1]
b2 = break2_list[-1]
e3 = end3_list[-1]

# === Step 1: Build last row point IDs ===
seg1_ids = [b1-3, b1-4] + list(range(b1-2, b1+1))
seg2_ids = list(range(b1, b2+1))
seg3_ids = list(range(b2, e3+1))

all_lastrow_ids = seg1_ids + seg2_ids + seg3_ids

# === Step 2: Create Point A (ymax, zmin) ===
xA = points[start1_list[-1] - 1][0]
pointA = (xA, "ymax", "zmin")

# === Step 3: Create Point B (ymin, zmin) ===
xB = points[end3_list[-1] - 1][0]
pointB = (xB, "ymin", "zmin")

# === Step 4: Write to .geo ===
combined_ids = []  # keep track of new IDs

with open(geofile, "a") as f:
    new_id = start_point_id
    
    # Point A
    f.write(f"Point({new_id}) = {{{pointA[0]-x0}, {pointA[1]}, {pointA[2]}, dx}};\n")
    combined_ids.append(new_id)
    new_id += 1

    # Points from last row at zmin
    for pid in all_lastrow_ids:
        x, y, _ = points[pid - 1]
        f.write(f"Point({new_id}) = {{{x-x0}, {y-y0}, zmin, dx}};\n")
        combined_ids.append(new_id)
        new_id += 1

    # Point B
    f.write(f"Point({new_id}) = {{{pointB[0]-x0}, {pointB[1]}, {pointB[2]}, dx}};\n")
    combined_ids.append(new_id)
    new_id += 1

print("Combined new point IDs:", combined_ids[:5], "... total:", len(combined_ids))

start_point_id = new_id  # Update start_point_id for next use

In [ ]:
# Assuming `combined_ids` is [A, ...all_lastrow..., B]
A_id = combined_ids[0]
B_id = combined_ids[-1]

# Extract IDs for last row points (zmin)
lastrow_new_ids = combined_ids[1:-1]

# Map old segX_ids to their new zmin counterparts
seg1_new_ids = [lastrow_new_ids[all_lastrow_ids.index(pid)] for pid in seg1_ids]
seg2_new_ids = [lastrow_new_ids[all_lastrow_ids.index(pid)] for pid in seg2_ids]
seg3_new_ids = [lastrow_new_ids[all_lastrow_ids.index(pid)] for pid in seg3_ids]

with open(geofile, "a") as f:
    # 1) Line: A -> first of seg1
    start_line_id += 1
    f.write(f"Line({start_line_id}) = {{{A_id}, {seg1_new_ids[0]}}};\n")

    # 2) Spline: seg1
    start_line_id += 1
    f.write(f"Spline({start_line_id}) = {{{', '.join(map(str, seg1_new_ids))}}};\n")

    # 3) Spline: seg2
    start_line_id += 1
    f.write(f"Spline({start_line_id}) = {{{', '.join(map(str, seg2_new_ids))}}};\n")

    # 4) Spline: seg3
    start_line_id += 1
    f.write(f"Spline({start_line_id}) = {{{', '.join(map(str, seg3_new_ids))}}};\n")

    # 5) Line: last of seg3 -> B
    start_line_id += 1
    f.write(f"Line({start_line_id}) = {{{seg3_new_ids[-1]}, {B_id}}};\n")

In [ ]:
# Read in the trench location, add them in at Z=0
trench_file = datadir + "trench_xyz.txt"
# Read space-delimited text into DataFrame
trench = pd.read_csv(trench_file, 
                        delim_whitespace=True,  # space-delimited
                        header=None,            # no header in file
                        names=["x", "y"])       # assign column names
trench = trench[(trench["y"] >= 120e3) & (trench["y"] <= 460e3)].reset_index(drop=True)
print(trench.head())




In [ ]:
# Sort the extracted data by x-values for consistency
trench = trench.sort_values(by='y', ascending=False).reset_index(drop=True)    

# Add a top boundary point at ymax
t_bound = pd.DataFrame([{'x': trench['x'].iloc[0], 'y': ymax, 'z': zmax}])
trench = pd.concat([t_bound, trench], axis=0, ignore_index=True)

# Add a bottom boundary point at ymin
b_bound = pd.DataFrame([{'x': trench['x'].iloc[-1], 'y': ymin, 'z': zmax}])
trench = pd.concat([trench, b_bound], axis=0, ignore_index=True)

# Sort the extracted data by x-values for consistency
trench = trench.sort_values(by='y', ascending=False).reset_index(drop=True)    

# for the surface, ie., z=0, use the coarse trench data
trench_points = []
point_indices = []  # Store indices for the current depth

# Format and append the first point with fine resolution
tmp = trench.iloc[0]
point_indices.append(start_point_id)  # Store the index for this depth
trench_points.append(f"Point({start_point_id}) = {{{tmp.x-x0:.6f}, {tmp.y:.6f}, zmax, dx}};\n")

# Loop over the middle points and format them with standard fault resolution
for i, row in enumerate(trench.iloc[1:-1].itertuples(index=False), start=1):
    start_point_id +=1
    point_indices.append(start_point_id)
    trench_points.append(f"Point({start_point_id}) = {{{row.x-x0:.6f}, {row.y-y0:.6f}, zmax, dx_fault_coarse}};\n")

# Format and append the last point with fine resolution
tmp = trench.iloc[-1]
start_point_id +=1
point_indices.append(start_point_id)
trench_points.append(f"Point({start_point_id}) = {{{tmp.x-x0:.6f}, {tmp.y:.6f}, zmax, dx}};\n")

# # Loop over with standard dx resolution
# for i, row in enumerate(trench.itertuples(index=False), start=1):
#     point_index = start_index + i - 1  # Compute the actual index
#     point_indices.append(point_index)  # Store the index for this depth
#     trench_points.append(f"{sep}\nPoint({point_index}) = {{{row.x:.6f}, {row.y:.6f}, {row.z:.6f}, dx}};")

# Append formatted points to the output .geo file
with open(geofile, "a") as f:
    # f.write("\n".join(trench_points) + "\n")
    f.writelines(trench_points)

# Increment the start index for the next set of points
print(start_point_id)

In [ ]:
# Define the 8 corner points
ncorners = 8
corner_points = [
    'xmin, ymax, zmax, dx', 
    'xmax, ymax, zmax, dx', 
    'xmax, ymin, zmax, dx', 
    'xmin, ymin, zmax, dx', 
    'xmin, ymax, zmin, dx',
    'xmax, ymax, zmin, dx',
    'xmax, ymin, zmin, dx',
    'xmin, ymin, zmin, dx',
]

# Append boundary points to the geo file
# start_point_id = npts + 1  # Start from the next point ID after the last point
with open(geofile, "a") as f:
    for i, coords in enumerate(corner_points, start=start_point_id+1):
        f.write(f"{sep}\nPoint({i}) = {{{coords}}};\n")
start_point_id += ncorners    
    

In [ ]:
# # add boundary points at ymin
# with open(geofile, "a") as f:
#     for i, pid in enumerate(end3_list, start=start_point_id):
#         x, _, z = points[pid - 1]  # pid is 1-based, array is 0-based
#         f.write(f"Point({i}) = {{{x-130000}, ymin, {z}, dx}};\n")

# start_point_id += len(start1_list) 